# Linear Regression Pairs Trading — XOM vs CVX

**Fixes applied vs original:**
- Corrected data files to actual XOM / CVX data
- Added Engle-Granger cointegration test and ADF stationarity checks
- Fixed look-ahead bias: hedge ratio fit on training window only
- Fixed reversed signal logic (original had Long/Short backwards)
- Fixed broken exit signal (original condition was logically impossible)
- Replaced fake random hedge-ratio plot with real static beta line
- Added transaction costs
- Added SOFR-approximated risk-free rate to Sharpe calculation
- Added max drawdown and win-rate metrics
- Clean train / test split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.stattools import coint, adfuller
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
# FIX: original code loaded f_with_log.csv (Ford) and gm_with_log.csv (GM)
# but named the variables xom_data / cvx_data. Using the correct XOM/CVX files.
xom_data = pd.read_csv('xom_with_log.csv')
cvx_data = pd.read_csv('cvx_with_log.csv')

xom_data['Date'] = pd.to_datetime(xom_data['Date'])
cvx_data['Date'] = pd.to_datetime(cvx_data['Date'])

# Recompute log close in case the stored column has NaNs
xom_data['Log_Close'] = np.log(xom_data['Close'])
cvx_data['Log_Close'] = np.log(cvx_data['Close'])

merged = pd.merge(
    xom_data[['Date', 'Log_Close']],
    cvx_data[['Date', 'Log_Close']],
    on='Date', suffixes=('_XOM', '_CVX')
).sort_values('Date').reset_index(drop=True)

print(f'Total observations : {len(merged)}')
print(f'Date range         : {merged.Date.min().date()} to {merged.Date.max().date()}')
merged.head()

## 2. Train / Test Split

**Critical**: the hedge ratio must be estimated on training data only.  
Using future data to fit beta is look-ahead bias and inflates backtest returns.

In [ ]:
TRAIN_START = '2010-01-01'
TRAIN_END   = '2013-12-31'
TEST_END    = '2014-12-31'

train = merged[(merged.Date >= TRAIN_START) & (merged.Date <= TRAIN_END)].copy()
test  = merged[(merged.Date >  TRAIN_END)   & (merged.Date <= TEST_END)].copy()

print(f'Train: {len(train)} observations  ({train.Date.min().date()} – {train.Date.max().date()})')
print(f'Test : {len(test)}  observations  ({test.Date.min().date()}  – {test.Date.max().date()})')

## 3. Stationarity & Cointegration Tests

Pairs trading requires:
1. Each individual series to be **non-stationary (I(1))**  
2. A **linear combination** (the spread) to be **stationary**  

The Engle-Granger test checks both simultaneously.  
We run it on **training data only** to avoid data snooping.

In [ ]:
# --- ADF on individual log-price series (should be non-stationary / unit root) ---
for label, series in [('XOM', train.Log_Close_XOM), ('CVX', train.Log_Close_CVX)]:
    stat, pval, _, _, crit, _ = adfuller(series, autolag='AIC')
    verdict = 'Non-stationary (I(1)) ✓' if pval > 0.05 else 'Stationary — unexpected'
    print(f'ADF {label}: stat={stat:.4f}  p={pval:.4f}  → {verdict}')

print()

# --- Engle-Granger cointegration test ---
t_stat, p_coint, crit_vals = coint(train.Log_Close_XOM, train.Log_Close_CVX)
print('Engle-Granger Cointegration Test (training period)')
print(f'  t-stat      : {t_stat:.4f}')
print(f'  p-value     : {p_coint:.4f}')
print(f'  Crit values : 1%={crit_vals[0]:.4f}  5%={crit_vals[1]:.4f}  10%={crit_vals[2]:.4f}')
if p_coint < 0.05:
    print('  → COINTEGRATED at 5% — pairs trading is statistically justified ✓')
else:
    print('  → WARNING: Not cointegrated at 5% — reconsider this pair')

## 4. Fit Hedge Ratio on Training Data Only

In [ ]:
# FIX: original code fit on the full merged dataset then filtered to 2010-2013.
# That leaks post-2013 price data into the hedge ratio estimate.
X_train = train.Log_Close_XOM.values.reshape(-1, 1)
y_train = train.Log_Close_CVX.values

model = LinearRegression()
model.fit(X_train, y_train)

alpha = model.intercept_
beta  = model.coef_[0]
r2    = model.score(X_train, y_train)

print(f'Hedge Ratio β  : {beta:.4f}')
print(f'Intercept α    : {alpha:.4f}')
print(f'R²             : {r2:.4f}')

# Compute spread on training set
train['Spread'] = train.Log_Close_CVX - (beta * train.Log_Close_XOM + alpha)

# Verify spread stationarity
stat, pval, *_ = adfuller(train.Spread.dropna(), autolag='AIC')
verdict = 'Stationary ✓' if pval < 0.05 else 'Non-stationary — WARNING'
print(f'\nADF on spread: stat={stat:.4f}  p={pval:.4f}  → {verdict}')

## 5. Rolling Z-Score & Signal Generation

**Signal logic (was reversed in original):**
- Spread HIGH (Z > +threshold): CVX expensive relative to XOM → **Short CVX / Long XOM**  
- Spread LOW  (Z < −threshold): CVX cheap relative to XOM   → **Long CVX / Short XOM**  
- |Z| < exit threshold: close position

In [ ]:
WINDOW   = 30   # rolling window for z-score
ENTRY_Z  = 1.0  # open position when |z| exceeds this
EXIT_Z   = 0.25 # close position when |z| falls below this

train['Rolling_Mean'] = train.Spread.rolling(WINDOW).mean()
train['Rolling_Std']  = train.Spread.rolling(WINDOW).std()
train['Z_Score']      = (train.Spread - train.Rolling_Mean) / train.Rolling_Std

# FIX: signals were reversed; Long fires on LOW z-score, Short on HIGH z-score
train['Long_Signal']  = train.Z_Score < -ENTRY_Z   # buy spread (long CVX, short XOM)
train['Short_Signal'] = train.Z_Score >  ENTRY_Z   # sell spread (short CVX, long XOM)

print(f'Long signals  : {train.Long_Signal.sum()}')
print(f'Short signals : {train.Short_Signal.sum()}')

## 6. Position Carry-Forward (Iterative State Machine)

Positions must be built **row-by-row** to respect causality.  
The original code mixed vectorised assignment with a loop, causing incorrect carry-forward.

In [ ]:
z = train.Z_Score.values
positions = np.zeros(len(train))
current   = 0

for i in range(len(train)):
    if np.isnan(z[i]):
        positions[i] = 0
        continue

    if current == 0:
        if z[i] < -ENTRY_Z:
            current = 1   # enter long spread
        elif z[i] > ENTRY_Z:
            current = -1  # enter short spread
    elif current == 1 and z[i] > -EXIT_Z:
        current = 0       # exit long (spread reverted above exit threshold)
    elif current == -1 and z[i] < EXIT_Z:
        current = 0       # exit short

    positions[i] = current

train['Position'] = positions
print(f'Long positions  : {(train.Position == 1).sum()} days')
print(f'Short positions : {(train.Position == -1).sum()} days')
print(f'Flat            : {(train.Position == 0).sum()} days')

## 7. Strategy Returns & Performance Metrics

In [ ]:
# Transaction cost: 1 bp round-trip per trade (conservative for liquid large-caps)
TC = 0.0001

train['Spread_Return']   = train.Spread.diff()
train['Trades']          = train.Position.diff().abs()
train['Strategy_Return'] = (
    train.Position.shift(1) * train.Spread_Return
    - train.Trades * TC
)

# Cumulative return: use cumprod, NOT cumsum
train['Cum_Return'] = (1 + train.Strategy_Return.fillna(0)).cumprod()

# --- Metrics ---
daily = train.Strategy_Return.dropna()
TRADING_DAYS = 252

# Approximate SOFR for 2010-2013 era (~0.05-0.25% annual → ~0.001% daily)
RF_DAILY = 0.00001
excess   = daily - RF_DAILY
sharpe   = excess.mean() / excess.std() * np.sqrt(TRADING_DAYS)

# Max drawdown
rolling_max = train.Cum_Return.cummax()
drawdown    = (train.Cum_Return - rolling_max) / rolling_max
max_dd      = drawdown.min()

# Win rate (only on trading days)
trading_returns = daily[daily != 0]
win_rate = (trading_returns > 0).mean()

total_return = train.Cum_Return.iloc[-1] - 1
total_trades = int(train.Trades.sum())

print('=' * 42)
print('  TRAINING PERIOD PERFORMANCE SUMMARY')
print('=' * 42)
print(f'  Cumulative Return  : {total_return*100:>8.2f}%')
print(f'  Sharpe Ratio       : {sharpe:>8.2f}')
print(f'  Max Drawdown       : {max_dd*100:>8.2f}%')
print(f'  Win Rate           : {win_rate*100:>8.1f}%')
print(f'  Total Trades       : {total_trades:>8d}')
print(f'  Transaction Cost   : {TC*1e4:.1f} bp per trade')

## 8. Visualisations

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=True)

# --- Panel 1: Log prices ---
ax = axes[0]
ax.plot(train.Date, train.Log_Close_CVX, label='CVX', alpha=0.8)
ax.plot(train.Date, train.Log_Close_XOM, label='XOM', alpha=0.8)
ax.set_title('Logarithmic Close Prices: CVX vs XOM', fontsize=13)
ax.set_ylabel('Log Price')
ax.legend(); ax.grid(True, alpha=0.3)

# --- Panel 2: Spread + Bollinger bands ---
ax = axes[1]
ax.plot(train.Date, train.Spread, label='Spread', color='green', lw=1, alpha=0.8)
ax.plot(train.Date, train.Rolling_Mean, label='Rolling Mean', color='orange', lw=1.2)
ax.plot(train.Date, train.Rolling_Mean + ENTRY_Z * train.Rolling_Std,
        color='red', linestyle='--', alpha=0.7, label=f'+{ENTRY_Z}σ')
ax.plot(train.Date, train.Rolling_Mean - ENTRY_Z * train.Rolling_Std,
        color='blue', linestyle='--', alpha=0.7, label=f'-{ENTRY_Z}σ')
long_pts  = train[train.Long_Signal]
short_pts = train[train.Short_Signal]
ax.scatter(long_pts.Date,  long_pts.Spread,  color='blue',  marker='^', s=40, zorder=5, label='Long Entry')
ax.scatter(short_pts.Date, short_pts.Spread, color='red',   marker='v', s=40, zorder=5, label='Short Entry')
ax.set_title('Spread and Trading Signals', fontsize=13)
ax.set_ylabel('Spread')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# --- Panel 3: Position ---
ax = axes[2]
ax.fill_between(train.Date, train.Position, alpha=0.5,
                where=train.Position > 0, color='blue',  label='Long Spread (long CVX, short XOM)')
ax.fill_between(train.Date, train.Position, alpha=0.5,
                where=train.Position < 0, color='red',   label='Short Spread (short CVX, long XOM)')
ax.axhline(0, color='black', lw=0.5)
ax.set_title('Position Over Time', fontsize=13)
ax.set_ylabel('Position')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# --- Panel 4: Cumulative return ---
ax = axes[3]
ax.plot(train.Date, train.Cum_Return, label='Pairs Strategy', color='navy', lw=1.5)
ax.axhline(1.0, color='black', linestyle='--', alpha=0.4, label='Break-even')
ax.set_title('Cumulative Returns (with transaction costs)', fontsize=13)
ax.set_ylabel('Cumulative Return')
ax.set_xlabel('Date')
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../docs/cum_LR_return.jpeg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Log close prices subplot (saved separately for website)
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.Date, train.Log_Close_CVX, label='CVX Log Close', alpha=0.8)
ax.plot(train.Date, train.Log_Close_XOM, label='XOM Log Close', alpha=0.8)
ax.set_title('Logarithmic Close Prices: CVX vs XOM', fontsize=14)
ax.set_xlabel('Date'); ax.set_ylabel('Log Price')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/logcloseprices_LR.jpeg', dpi=150, bbox_inches='tight')
plt.show()

# Spread + signals subplot (saved separately)
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(train.Date, train.Log_Close_CVX, label='CVX', alpha=0.8)
axes[0].plot(train.Date, train.Log_Close_XOM, label='XOM', alpha=0.8)
axes[0].set_title('Log Prices: CVX vs XOM', fontsize=13)
axes[0].set_ylabel('Log Price'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(train.Date, train.Spread, label='Spread', color='green', alpha=0.7, lw=1)
axes[1].plot(train.Date, train.Rolling_Mean, label='Rolling Mean', color='orange')
axes[1].plot(train.Date, train.Rolling_Mean + ENTRY_Z * train.Rolling_Std,
             color='red', linestyle='--', alpha=0.7)
axes[1].plot(train.Date, train.Rolling_Mean - ENTRY_Z * train.Rolling_Std,
             color='blue', linestyle='--', alpha=0.7)
axes[1].scatter(long_pts.Date,  long_pts.Spread,  color='blue', marker='^', s=40, label='Buy Signal')
axes[1].scatter(short_pts.Date, short_pts.Spread, color='red',  marker='v', s=40, label='Sell Signal')
axes[1].set_title('Spread and Trading Signals', fontsize=13)
axes[1].set_ylabel('Spread'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/logprices_LR.jpeg', dpi=150, bbox_inches='tight')
plt.show()

## 9. Out-of-Sample Test (2014)

Apply the **same** hedge ratio (β, α) learned from training to unseen data.

In [ ]:
if len(test) > 0:
    test = test.copy()
    test['Spread']        = test.Log_Close_CVX - (beta * test.Log_Close_XOM + alpha)
    test['Rolling_Mean']  = test.Spread.rolling(WINDOW).mean()
    test['Rolling_Std']   = test.Spread.rolling(WINDOW).std()
    test['Z_Score']       = (test.Spread - test.Rolling_Mean) / test.Rolling_Std

    z_test    = test.Z_Score.values
    pos_test  = np.zeros(len(test))
    current   = 0
    for i in range(len(test)):
        if np.isnan(z_test[i]):
            pos_test[i] = 0; continue
        if current == 0:
            if z_test[i] < -ENTRY_Z:   current = 1
            elif z_test[i] > ENTRY_Z:  current = -1
        elif current == 1  and z_test[i] > -EXIT_Z:  current = 0
        elif current == -1 and z_test[i] <  EXIT_Z:  current = 0
        pos_test[i] = current

    test['Position']        = pos_test
    test['Spread_Return']   = test.Spread.diff()
    test['Trades']          = test.Position.diff().abs()
    test['Strategy_Return'] = test.Position.shift(1) * test.Spread_Return - test.Trades * TC
    test['Cum_Return']      = (1 + test.Strategy_Return.fillna(0)).cumprod()

    oos_daily  = test.Strategy_Return.dropna()
    oos_excess = oos_daily - RF_DAILY
    oos_sharpe = oos_excess.mean() / oos_excess.std() * np.sqrt(TRADING_DAYS) if oos_excess.std() > 0 else 0
    oos_return = test.Cum_Return.iloc[-1] - 1
    oos_trades = int(test.Trades.sum())
    oos_max_dd = ((test.Cum_Return - test.Cum_Return.cummax()) / test.Cum_Return.cummax()).min()

    print('=' * 42)
    print('  OUT-OF-SAMPLE PERFORMANCE (2014)')
    print('=' * 42)
    print(f'  Cumulative Return  : {oos_return*100:>8.2f}%')
    print(f'  Sharpe Ratio       : {oos_sharpe:>8.2f}')
    print(f'  Max Drawdown       : {oos_max_dd*100:>8.2f}%')
    print(f'  Total Trades       : {oos_trades:>8d}')

    plt.figure(figsize=(12, 4))
    plt.plot(test.Date, test.Cum_Return, label='OOS Strategy', color='darkorange', lw=1.5)
    plt.axhline(1.0, color='black', linestyle='--', alpha=0.4)
    plt.title('Out-of-Sample Cumulative Returns (2014)', fontsize=13)
    plt.xlabel('Date'); plt.ylabel('Cumulative Return')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No out-of-sample data available in the current dataset.')

## Notes on Limitations

- **Static hedge ratio**: Beta is fixed for the entire backtest. Kalman filtering (see `kalman_filtering.ipynb`) adapts it dynamically.
- **No slippage model**: Transaction cost of 1 bp is conservative but real slippage may be higher for larger position sizes.
- **Survivorship bias**: XOM and CVX are both large-caps that survived the test period. Pre-selecting known survivors overstates expected performance.
- **Short window backtest**: 4 years is not sufficient to assess regime robustness. Walk-forward testing across multiple market cycles is recommended.